# Urgent Care Call Centre Model

Our model is a stylised model of a typical urgent care call centre. Patient calls arrive at random to a call centre. Operators answer the calls, with a first in first out queue, triage the patient, interview following a standard script, and provide a call designation; for example, if the patient should travel to an emergency department, book an appointment in primary care with a General Practitioner (family doctor), or if a call back from a nurse is needed. If a caller needs to speak to a nurse they enter a first in first out queue until a nurse is available. Figure 1 illustrates patient flow in the model and use of resources.

> Note the model does not include features like reneging, time dependency, or shift work.

<img src="images/call_centre_diagram.png" alt="Call Centre Diagram" width="600">

## 1. Imports

In [1]:
import simpy

# import the model and the scenario parameter class
from urgent_care_sim import (
    Scenario, 
    UrgentCareCallCentre, 
    set_trace,
    Auditor,
    run_results
)

## 2. How to create a simpy environment

In [2]:
env = simpy.Environment()

## 3. Create a default scenario 

After we create an instance of a `simpy.Environment` we should create a `Scenario`. This is a parameter class. It holds, for example, the number of call operators and nurses on duty.

In [3]:
env = simpy.Environment()
default_args = Scenario(env)

In [4]:
default_args

Scenario(
  main_seed=0,
  n_operators=13, n_nurses=10,
  mean_iat=0.60,
  call_dist(low=5, mode=7, high=10),
  nurse_dist(low=10.0, high=20.0),
  p_callback=0.4
)

We create a different scenario by setting the parameters

In [5]:
env = simpy.Environment()
extra_nurse_scenario_args = Scenario(env, n_nurses=11)
extra_nurse_scenario_args

Scenario(
  main_seed=0,
  n_operators=13, n_nurses=11,
  mean_iat=0.60,
  call_dist(low=5, mode=7, high=10),
  nurse_dist(low=10.0, high=20.0),
  p_callback=0.4
)

## 4. How to create an instance of a call centre model

In this example, the model is implemented in a class.  It could also be a function.  The class is called `UrgentCareCallCentre`. We need to pass it the `simpy.Environment` and the `Scenario` we have created.

In [6]:
env = simpy.Environment()

# create a default set of arguments - a "Scenario"
default_args = Scenario(env)

# create an instance of a model and pass it the default arguments
model = UrgentCareCallCentre(env, default_args)

## 5. A example script to run the model

In [7]:
# create the simpy environment
env = simpy.Environment()

# create a default set of arguments - a "Scenario"
default_args = Scenario(env)

# create an instance of a model and pass it the default arguments
model = UrgentCareCallCentre(env, default_args)

# setup the arrivals generator as a simpy process 
# this will simulate calls and simulate patients progress 
# through the call centre
env.process(model.arrivals_generator())

# show a "trace" of patients and their progress through the model
# setting to True is helpful for debugging.
set_trace(False)

# run for 25 mins
env.run(until=25)

print(f'end of run. simulation clock time = {env.now}')

Simulation tracing set to: False
end of run. simulation clock time = 25


## 6. How to get results out of the model

Here we will run the model for 1000 minutes and record data as we go along.

In [9]:
# create the simpy environment
env = simpy.Environment()

# create a default set of arguments - a "Scenario"
default_args = Scenario(env)

# create an instance of a model and pass it the default arguments
model = UrgentCareCallCentre(env, default_args)

# create an auditor that will take observations of queue length every
# 5 minutes.
auditor = Auditor(
    env=env, 
    run_length=1000, 
    first_obs=5, 
    interval=5
)

# add the resources to the auditor
auditor.add_resource_to_audit(default_args.operators, 'ops')
auditor.add_resource_to_audit(default_args.nurses, 'nurse')

# setup the arrivals generator as a simpy process 
# this will simulate calls and simulate patients progress 
# through the call centre
env.process(model.arrivals_generator())

# show a "trace" of patients and their progress through the model
# setting to True is helpful for debugging.
set_trace(False)

# run for 1000 mins
env.run(until=1000)


print(f'end of run. simulation clock time = {env.now}')

# get the results of the model run (returns a pandas dataframe)
run_results(model, auditor).round(2)

Simulation tracing set to: False
end of run. simulation clock time = 1000


,estimate
mean_queue_ops,6.53
mean_queue_nurse,27.28
mean_system_ops,18.70
mean_system_nurse,37.14
mean_wait,3.87
ops_util,0.94
mean_nurse_wait,37.72
nurse_util,0.99
